In [10]:
from PIL import Image
import math
import random

5.      На основе одного из реализованных алгоритмов закраски замкнутых областей выполнить раскраску полигонов, составляющих «проволочный» рендер заданной 3D модели, случайными цветами.

In [11]:
def project(vertex):
    x = int((vertex[0]) * 150)
    y = int(vertex[1] * 150)
    return x, y
def y_list(vertices, face):
    y_table = [[] for i in range(501)]
    for i in range(len(face)):
        v1 = vertices[face[i]]
        v2 = vertices[face[(i + 1) % len(face)]]
        x0, y0 = project(v1)
        x1, y1 = project(v2)
        if y0 == y1: continue
        if y0 < y1: x0, y0, x1, y1 = x1, y1, x0, y0
        y_table[250 - y0].append([250 - y1, 250 + x0, (x1 - x0) / (y0 - y1)])
    return y_table
def cat(img, y_table):
    w, h = img.size
    pixels = img.load()
    aet = []
    color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
    for y_current in range(501):
        aet.extend(y_table[y_current])
        aet = [edge for edge in aet if y_current < edge[0]]
        aet.sort(key=lambda edge: edge[1])
        for i in range(0, len(aet) - 1, 2):
            x_start = math.ceil(aet[i][1])
            x_end = math.floor(aet[i + 1][1])
            for j in range(x_start, x_end + 1):
                pixels[j, y_current] = color
        for i in range(len(aet)):
            aet[i][1] += aet[i][2]
def alg_4(vertices, face, img):
    y_table = y_list(vertices, face)
    cat(img, y_table)
def render(file):
    vertices = []
    faces = []
    with open(file, 'r') as f:
        for line in f:
            if line.startswith('v '):
                vertices.append(list(map(float, line.split()[1:4])))
            elif line.startswith('f '):
                faces.append([int(i.split('/')[0]) - 1 for i in line.split()[1:]])
    img = Image.new('RGB', (501, 501), color=(255, 255, 255))
    for face in faces:
        alg_4(vertices, face, img)
    img.show()

In [12]:
render("african_head.obj")